# Pipeline Inti SF3D (Jobdesk 1 / Ketua) - Tahap A–E

Foto artefak → GLB bertekstur, otomatis. Runtime: **Colab/Kaggle GPU T4**.

Urutan: setup → login HF (model gated) → muat SF3D → inferensi batch → cek VRAM/timing → serah-terima GLB ke `outputs/`.

> Pastikan Runtime > Change runtime type > **T4 GPU** sebelum jalan.

In [ ]:
#@title 1. Setup environment
!nvidia-smi -L
!git clone https://github.com/Stability-AI/stable-fast-3d.git sf3d_repo
%cd sf3d_repo
!pip install -q -r requirements.txt
!pip install -q ./texture_baker ./uv_unwrapper
!pip install -q omegaconf trimesh rembg onnxruntime
%cd ..

In [ ]:
#@title 2. Login Hugging Face (accept license SF3D gated dulu di web)
from huggingface_hub import login
login()  # tempel token. Wajib sudah klik 'Agree' di halaman model SF3D.

In [ ]:
#@title 3. Clone repo proyek (kode pipeline tim)
REPO_URL = ""  #@param {type:"string"}  # URL repo bersama tim
import os
if REPO_URL and not os.path.exists("FP_GRAFKOM"):
    !git clone $REPO_URL FP_GRAFKOM
import sys
sys.path.insert(0, "sf3d_repo")            # paket sf3d
sys.path.insert(0, "FP_GRAFKOM/pipeline/src")  # kode pipeline tim

In [ ]:
#@title 4. Muat model SF3D sekali (pakai ulang untuk batch)
from config import PipelineCfg
from inference import SF3DReconstructor
from preprocess import load_rembg_session

cfg = PipelineCfg.load("FP_GRAFKOM/pipeline/configs/default.yaml",
                       **{"export.out_dir": "FP_GRAFKOM/outputs"})
reconstructor = SF3DReconstructor(cfg.model)
rembg_session = load_rembg_session()
print("device:", reconstructor.device)

In [ ]:
#@title 5. Inferensi batch tahap A–E
from run_pipeline import collect_images, process_one
from export_glb import write_manifest
from pathlib import Path

INPUT = "FP_GRAFKOM/dataset/photos"  #@param {type:"string"}
TIER = "high"  #@param ["high", "mid", "low"]
manifest = Path(cfg.export.out_dir) / "manifest.jsonl"

for img in collect_images(Path(INPUT)):
    try:
        rec = process_one(img, reconstructor, cfg, rembg_session, TIER)
        write_manifest(rec, manifest)
        print(f"{img.name} -> {Path(rec['glb']).name} | {rec['seconds']}s | "
              f"VRAM {rec['peak_vram_gb']}GB | {rec['glb_bytes']//1024}KB")
    except Exception as e:
        print("GAGAL", img.name, e)

In [ ]:
#@title 6. Verifikasi material PBR & delighting (tanggung jawab ketua)
# Render cepat satu mesh utk cek albedo/roughness/metallic tidak hangus.
import trimesh, numpy as np
from pathlib import Path
glb = sorted(Path(cfg.export.out_dir, "glb").glob("*.glb"))[0]
scene = trimesh.load(glb)
print("loaded:", glb.name)
print("geometri:", {k: (v.vertices.shape, v.faces.shape) for k, v in scene.geometry.items()})
for name, geom in scene.geometry.items():
    mat = getattr(geom.visual, 'material', None)
    print(name, "material:", type(mat).__name__,
          "| has baseColorTexture:", getattr(mat, 'baseColorTexture', None) is not None)

## Serah terima
- `outputs/glb/*.glb` → Jobdesk 3 (post-process, kompresi Draco/KTX2, viewer).
- `outputs/raw_mesh/*` + `outputs/manifest.jsonl` → Jobdesk 4 (evaluasi).
- Commit `outputs/` ke repo rutin (sesi Colab bisa putus, PRD §14).